In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rushibalajiputthewad/sound-classification-of-animal-voice")

print("Path to dataset files:", path)

100%|██████████| 37.7M/37.7M [00:01<00:00, 38.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/rushibalajiputthewad/sound-classification-of-animal-voice/versions/1


In [2]:
import torch
import torchaudio
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as transforms

In [3]:
test_sound_path = "/root/.cache/kagglehub/datasets/rushibalajiputthewad/sound-classification-of-animal-voice/versions/1/Animal-Soundprepros/Bear/Bear_2.wav"
waveform, sample = torchaudio.load(test_sound_path)
waveform.shape

torch.Size([2, 19522])

In [43]:
#resize audio
transform = transforms.Resample(sample, int(sample/4))

In [38]:
test_sound_path2 = "/root/.cache/kagglehub/datasets/rushibalajiputthewad/sound-classification-of-animal-voice/versions/1/Animal-Soundprepros/Bear/Bear_1.wav"
waveform2, sample2 = torchaudio.load(test_sound_path)
waveform2 = transform(waveform2)
waveform2.shape

torch.Size([2, 3905])

In [39]:
waveform = transform(waveform)
waveform.shape

torch.Size([2, 35])

In [40]:
import pandas as pd

csv_file = pd.read_csv("/root/.cache/kagglehub/datasets/rushibalajiputthewad/sound-classification-of-animal-voice/versions/1/Animal_Sound.csv")
len(csv_file["path"])

650

In [34]:
#iterate through all folders to get all folder paths because
import os

def get_file_paths(path):
  file_paths = []
  for root, dirs, files in os.walk(path):
    for name in files:
      if ".wav" in name:
        file_paths.append(os.path.join(root, name))
  return file_paths

file_paths = get_file_paths("/root/.cache/kagglehub/datasets/rushibalajiputthewad/sound-classification-of-animal-voice/versions/1")
print(file_paths[100])

/root/.cache/kagglehub/datasets/rushibalajiputthewad/sound-classification-of-animal-voice/versions/1/Animal-Soundprepros/Bear/Bear_9.wav


# Dataset and Dataloader

In [48]:
#resize audio
transform = transforms.Resample(sample, int(sample/4))

In [70]:
from torchvision.datasets import DatasetFolder
import torch.nn.functional as F

def audio_loader(path):
  waveform, sample = torchaudio.load(path, num_frames=1952, channels_first=True)
  waveform = transform(waveform)
  if waveform.shape[0] != 1:
    waveform = waveform[0:1, :]

  if waveform.shape[1] > 488:
    waveform = waveform[0:1, :488]
  elif waveform.shape[1] < 488:
    waveform  = F.pad(waveform, (0, 488 - waveform.shape[1]))

  return waveform
  #waveform.shape


dataset = DatasetFolder("/root/.cache/kagglehub/datasets/rushibalajiputthewad/sound-classification-of-animal-voice/versions/1/Animal-Soundprepros",
                        loader=audio_loader,
                        extensions=('.wav', '.mp3')
                        )
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)
train_loader

In [71]:
sound, label = next(iter(train_loader))
label
sound.shape

torch.Size([8, 1, 488])

# **MODEL**

In [72]:
if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"

In [73]:
class AudioClassifier(nn.Module):
  def __init__(self, input_size, hidden_size, class_num):
    super().__init__()
    self.conv_block1 = nn.Sequential(
        nn.Conv1d(input_size, 16, stride=2, kernel_size=2),
        nn.MaxPool1d(kernel_size=2, stride=2),
        nn.ReLU(),

        nn.Conv1d(16, 32, stride=2, kernel_size=2),
        nn.MaxPool1d(kernel_size=2, stride=2),
        nn.ReLU(),

        nn.Conv1d(32, 64, stride=2, kernel_size=2),
        nn.MaxPool1d(kernel_size=2, stride=2),
        nn.ReLU(),


        #bei maxpool stride=2 und kernel size=2 wird die witdh und height bei bswp conv2d immer halbiert
        nn.Conv1d(64, 128, stride=2, kernel_size=2),
        nn.MaxPool1d(kernel_size=2, stride=2),
        nn.ReLU(),

        nn.Flatten()
    )

    self.ffl = nn.Sequential(
        nn.Linear(128, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, class_num),
    )

  def forward(self, x):
    x = self.conv_block1(x)
    return self.ffl(x)

model = AudioClassifier(1, 200, 13).to(device)

In [74]:
#test model tinker tinker tinker

sound, label = next(iter(train_loader))
sound = sound.to(device)
out = model(sound)
out.shape

torch.Size([8, 13])

In [75]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [78]:
#test training loop tinker tinker tinker

epochs = 10

for epoch in range(epochs):
  total_loss = 0
  for sound, label in train_loader:
    sound = sound.to(device)

    out = model(sound)
    loss = loss_fn(out, label)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10, Loss: 2.4650
Epoch 2/10, Loss: 2.4460
Epoch 3/10, Loss: 2.4286
Epoch 4/10, Loss: 2.4148
Epoch 5/10, Loss: 2.4133
Epoch 6/10, Loss: 2.3855
Epoch 7/10, Loss: 2.3687
Epoch 8/10, Loss: 2.3708
Epoch 9/10, Loss: 2.3390
Epoch 10/10, Loss: 2.3387


In [83]:
sound, label = next(iter(train_loader))
sound = sound.to(device)
out = model(sound)

out = torch.softmax(out, dim=-1)
out = out.argmax(dim=-1)

print(out[0])
print(label[0])

tensor(10)
tensor(10)
